# 1. Construct a portfolio based on a 5 year data

In [12]:
from sp500 import (
    load_sp500_prices_df,
    compute_covariance,
    compute_log_returns,
    solve_sp500_fw_homotopy,
    solve_sp500_greedy,
)
import quantstats as qs

# PAST_START_DATE = "2018-01-01"
# PAST_END_DATE = "2024-01-01"
# TEST_START_DATE = "2024-02-01"
# TEST_END_DATE = "2026-01-01"
START_DATE = "2018-01-01"
END_DATE = "2026-03-30"
TRADING_DAYS = 252


prices_df = load_sp500_prices_df(start=START_DATE, end=END_DATE)
stock_names = prices_df.columns.tolist()


# Create benchmark df



[*******               15%                       ]  73 of 502 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: K"}}}
[*********************100%***********************]  502 of 502 completed

12 Failed downloads:
['JNPR', 'K', 'DFS', 'IPG', 'WBA', 'MMC', 'HES', 'ANSS', 'PARA', 'FI', 'DAY']: YFTzMissingError('possibly delisted; no timezone found')
['KDP']: Timeout('Failed to perform, curl: (28) Operation timed out after 10002 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


Saved 472 stocks x 2070 days → /Users/nautilus/gridfw/data/market/sp500_prices.csv


# Portfolio Construction
We first create a portfolio using a data from `2018-01-01` to `2023-01-01` (5 years worth data).

In [19]:
SAMPLE_START_DATE = "2018-01-01"
SAMPLE_END_DATE = "2023-01-01"
past_prices_df = prices_df.loc[SAMPLE_START_DATE:SAMPLE_END_DATE]


log_return_matrix = compute_log_returns(past_prices_df)
A = compute_covariance(log_return_matrix)
k = 50
fw_indices = solve_sp500_fw_homotopy(A=A, k=k, stock_names=stock_names)
greedy_indices = solve_sp500_greedy(A=A, k=k, stock_names=stock_names)


[FWHomotopy] p=472, k=50, steps=800, n_mc=100, alpha=0.01


In [23]:
TEST_START_DATE = "2023-03-01"
TEST_END_DATE = "2026-03-01"
# Mean across selected stocks → single portfolio return Series
future_prices_df = prices_df.loc[TEST_START_DATE:TEST_END_DATE]
fw_portfolio_returns = future_prices_df.iloc[:, fw_indices].pct_change().dropna().mean(axis=1)
greedy_portfolio_returns = future_prices_df.iloc[:, greedy_indices].pct_change().dropna().mean(axis=1)
# sp500_returns = future_prices_df.pct_change().dropna().mean(axis=1)
qs.reports.html(
    returns=fw_portfolio_returns,
    benchmark=greedy_portfolio_returns,
    output='fw_performance_report.html',
    title='FW-Homotopy Portfolio Performance',
    strategy_title = "FW-Homotopy",
    benchmark_title = "Greedy"
    
)


# ann_return = sp500_mean_returns.mean()
# Sharpe Ratio

# sp500_mean_returns
# fw_prices = prices_df.iloc[:, fw_indices]
# greedy_prices = prices_df.iloc[:, greedy_indices]

